In [1]:
// load the formatting setup and the style sheet
#load "../setup.fsx"
HTML(System.IO.File.ReadAllText "../style.css")

The below script needs to be able to find the current output cell; this is an easy method to get it.

# A simple Backtracking Constraint Solver

## Representing Constraint Satisfaction Problems

A *constraint satisfaction problem* (CSP) is a triple
$$ \mathcal{P} = \langle \mathtt{Variables}, \mathtt{Values}, \mathtt{Constraints} \rangle $$
where
- `Variables` is a list of strings which serve as *variables*,
- `Values` is a set of *values* that can be assigned to the variables,
- `Constraints` is a list of formulas.

A *variable assignment* is a map from variables to values.  In Python, the constraints are strings containing Python expressions.  These strings are evaluated using the function `eval`.  Furthermore, the Python version of the solver uses the module `ast` to find the variables that occur in a constraint.  F# is a compiled language and therefore has no function `eval`.  Instead, we represent a constraint as a *record* with three fields:
- `Variables` is the set of variables occurring in the constraint.
- `Holds` is a function that takes a variable assignment and returns `true` if the constraint is satisfied by this assignment.  This function is only called if all variables of the constraint have a value.
- `Text` is a string describing the constraint.  It is only used for displaying the constraint.

The type `Constraint<'V>` is *generic*: the type variable `'V` is the type of the values.  For example, for the map coloring problem the values are strings, while for the eight queens puzzle the values are integers.

In Python, the function `solve` first *annotates* every constraint with the set of variables occurring in it.  In F#, a constraint already contains this set.

In [2]:
type Constraint<'V> =
    { Variables : Set<string>
      Holds     : Map<string, 'V> -> bool
      Text      : string }
    override this.ToString() = this.Text

By default, a record is displayed by showing all its fields.  As the field `Holds` is a function, this is not helpful.  Therefore, we register a printer that displays a constraint by its text.  The function `registerShow` is defined in the file `setup.fsx`.  It takes a function that returns `Some s` if it can display the given object as the string `s`, and `None` otherwise.  The keyword `do` marks an expression that is evaluated for its effect.  When this notebook is imported by another notebook, cells starting with `do` are executed as well.

In [3]:
do registerShow (fun (o: obj) ->
    if not (isNull o) && o.GetType().Name = "Constraint`1" then Some (o.ToString()) else None)

The function `constr` takes three arguments:
- `text` is a string describing the constraint,
- `vars` is the list of the variables occurring in the constraint,
- `holds` is a function that checks whether the constraint is satisfied by a given variable assignment.

It returns the corresponding constraint.  For example, the constraint $x + y = 7 \cdot z$ is created as follows:
```
constr "x + y = 7 * z" ["x"; "y"; "z"] (fun A -> A["x"] + A["y"] = 7 * A["z"])
```
Here, `A` is the variable assignment and `A["x"]` is the value of the variable `x`.

In [4]:
let constr (text: string) (vars: string list) (holds: Map<string, 'V> -> bool) : Constraint<'V> =
    { Variables = set vars; Holds = holds; Text = text }

In [5]:
let c = constr "x + y = 7 * z" ["x"; "y"; "z"] (fun A -> A["x"] + A["y"] = 7 * A["z"])
c.Holds (Map [ ("x", 3); ("y", 4); ("z", 1) ])

true

Many constraints have the form $x \not= y$ or $x = y$.  The functions `different` and `equal` take two arguments:
- `x` is a variable,
- `y` is a variable.

They return the constraint $x \not= y$ and $x = y$, respectively.  The function `hasValue` takes a variable `x` and a value `v` and returns the constraint $x = v$.

In [6]:
let different (x: string) (y: string) : Constraint<'V> =
    constr (sprintf "%s ≠ %s" x y) [x; y] (fun A -> A[x] <> A[y])

let equal (x: string) (y: string) : Constraint<'V> =
    constr (sprintf "%s = %s" x y) [x; y] (fun A -> A[x] = A[y])

let hasValue (x: string) (v: 'V) : Constraint<'V> =
    constr (sprintf "%s = %A" x v) [x] (fun A -> A[x] = v)

A CSP is represented as a record with three fields.  The order of the variables in the list `Variables` determines the order in which values are assigned to the variables.

In [7]:
type CSP<'V> =
    { Variables   : string list
      Values      : 'V list
      Constraints : Constraint<'V> list }

## The Backtracking Solver

The function `isConsistent` takes four arguments:
- `var` is a variable.
- `value` is a value that is to be assigned to the variable `var`.
- `Assignment` is a partial variable assignment that does not assign a value for `var`
  and that is *consistent* with all constraints in the list `Constraints`.
- `Constraints` is a list of constraints.

The function checks whether the assignment
$$ \texttt{Assignment} \cup \{\texttt{var} \mapsto \texttt{value}\}$$
violates any of the constraints in `Constraints`.  It assumes that
`Assignment` is *consistent*.  Therefore, only those constraints have to be checked that contain the variable `var` and whose variables all have a value.

In [8]:
let isConsistent (var: string) (value: 'V) (Assignment: Map<string, 'V>) (Constraints: Constraint<'V> list) : bool =
    let NewA = Assignment.Add(var, value)
    Constraints |> List.forall (fun c ->
        if c.Variables.Contains var && c.Variables |> Set.forall NewA.ContainsKey then
            c.Holds NewA
        else
            true)

The function `backtrackSearch` takes two arguments:
- `Assignment` is a **consistent** *partial variable assignment*,
- `csp` is a constraint satisfaction problem.

It tries to extend the given assignment recursively and thereby produce a solution of the given CSP.  The result has the type `Map<string, 'V> option`: if a solution is found, it is returned as `Some Solution`.  Otherwise, `None` is returned.

**Implementation:**
1. If all variables are already assigned, then `Assignment` is a solution, since it is consistent.
2. Otherwise, we take the first variable `var` that has no value yet.  We try all values for this variable.  If the assignment `var ↦ value` is consistent with `Assignment`, we extend `Assignment` and try to solve the problem recursively.
3. `List.tryPick` applies a function to the elements of a list until the function returns a value of the form `Some x`.  This value is then returned.  If the function returns `None` for all elements of the list, the result is `None`.  Hence, the values are tried one after the other until a solution has been found.

In [9]:
let rec backtrackSearch (Assignment: Map<string, 'V>) (csp: CSP<'V>) : Map<string, 'V> option =
    // If all variables are already assigned, then Assignment is a solution
    // because it is consistent.
    if Assignment.Count = csp.Variables.Length then
        Some Assignment
    else
        // take first unassigned variable
        let var = csp.Variables |> List.find (fun x -> not (Assignment.ContainsKey x))
        // try all values for this variable
        csp.Values |> List.tryPick (fun value ->
            if isConsistent var value Assignment csp.Constraints then
                backtrackSearch (Assignment.Add(var, value)) csp
            else
                None)

The function `solve` takes one argument:
- `csp` is a constraint satisfaction problem.

It tries to compute a solution of this problem via *backtracking*.  The search starts with the empty variable assignment.

In [10]:
let solve (csp: CSP<'V>) : Map<string, 'V> option =
    backtrackSearch Map.empty csp

Let us test the solver with a small example.

In [11]:
let example : CSP<int> =
    { Variables   = [ "x"; "y"; "z" ]
      Values      = [ 1 .. 9 ]
      Constraints = [ constr "x + y = 7 * z" ["x"; "y"; "z"] (fun A -> A["x"] + A["y"] = 7 * A["z"])
                      different "x" "y"
                      constr "x > y" ["x"; "y"] (fun A -> A["x"] > A["y"]) ] }
solve example

Some map [("x", 4); ("y", 3); ("z", 1)]